### 1. Load the game data

- xml.etree.ElementTree: Tools for handling xml data
- ET.parse("save_file_name.rws"): It converts data in .rws to tree structure
- root: The root element of our tree structure

In [6]:
import xml.etree.ElementTree as ET

tree = ET.parse("logs/09_04_2026.rws")
root = tree.getroot()

### 2. Explore the XML structure I
- Print all tags from root down to 2 levels

In [2]:
print(f"{root.tag}/")
for child in root:
    print(f"    {child.tag}")

savegame/
    meta
    game


### 3. Explore the XML structure II
- Print all tags from root down to 3 levels

In [5]:
print(f"{root.tag}/")
for child in root:
    print(f"    {child.tag}/")
    for grandchild in child:
        print(f"        {grandchild.tag}")

savegame/
    meta/
        gameVersion
        modIds
        modSteamIds
        modNames
    game/
        currentMapIndex
        info
        rules
        scenario
        tickManager
        playSettings
        storyWatcher
        gameEnder
        letterStack
        researchManager
        analysisManager
        storyteller
        history
        taleManager
        playLog
        battleLog
        outfitDatabase
        drugPolicyDatabase
        foodRestrictionDatabase
        readingPolicyDatabase
        tutor
        dateNotifier
        uniqueIDsManager
        questManager
        transportShipManager
        studyManager
        customXenogermDatabase
        customXenotypeDatabase
        relationshipRecords
        hiddenItemsManager
        entityCodex
        components
        world
        maps
        cameraMap


### 4. Print all tags inside \<meta>
- Find the \<meta> tag and print all child tags and their values.

In [9]:
meta = root.find('meta')

for child in meta:
    if child.text is not None:
        print(f"{child.tag} -> {child.text}")
    else:
        print(f"{child.tag}")

gameVersion -> 1.6.4633 rev1269
modIds -> 
			
modSteamIds -> 
			
modNames -> 
			


Some tags contain \<li> elements instead of direct text, so I can't see anything after '->'. I will exlpore the contents for these \<li> tags in the next step.

### 5. Find the game version
- Find and print the \<gameVersion> value inside the \<meta> tag.
- Using **.text**

In [16]:
meta = root.find('meta')
gameVersion = meta.find('gameVersion')

print(gameVersion.text)

1.6.4633 rev1269


### 6. Find all unique tag names in \<game>
- Print all tag names that are direct children of \<game>. If there are identical tags, remove the duplicates.

In [12]:
game = root.find('game')

unique_tags = set()
for child in game:
    unique_tags.add(child.tag)

print(unique_tags)

{'analysisManager', 'researchManager', 'cameraMap', 'tickManager', 'customXenotypeDatabase', 'relationshipRecords', 'history', 'storyteller', 'transportShipManager', 'studyManager', 'dateNotifier', 'tutor', 'outfitDatabase', 'playSettings', 'world', 'playLog', 'battleLog', 'uniqueIDsManager', 'customXenogermDatabase', 'foodRestrictionDatabase', 'maps', 'hiddenItemsManager', 'gameEnder', 'storyWatcher', 'currentMapIndex', 'readingPolicyDatabase', 'scenario', 'letterStack', 'rules', 'entityCodex', 'drugPolicyDatabase', 'taleManager', 'info', 'questManager', 'components'}


Using **set()** for removing duplicates.

### 7. Find all mode names in 'meta/modNames'
- Using **.findall** for finding all nested paths.

In [23]:
modNames = root.find('meta/modNames')

# check what's in the modNames(here <li>)
# for child in modNames:
#     print(child.tag)

mod_list = meta.findall(".//modNames/li")
for mod in mod_list:
    print(mod.text)

Harmony
Core
Biotech
Interaction Bubbles
Research Whatever
RPG Style Inventory Revamped
XML Extensions
HugsLib
Map Preview
Prepare Landing (Continued)
Rimworld Mod Korean
[FSF] Filth Vanishes With Rain And Time
ConsumeInventoryItem together
Mark That Pawn
No Burn Metal
Tilled Soil
Pick Up And Haul
Quick Stockpile Creation
RandomPlus
EdB Prepare Carefully
Replace Stuff - Continued
RimHUD
Smart Speed
Smarter Construction
Useful Marks


### 8. Find number of maps in 'game/maps'
- Find and print how many children in 'game/maps/li', and print all child tags of first child.

In [ ]:
game = root.find('game')

# check what's in maps
maps = game.find('maps')
print(f"{maps.tag}/")
for child in maps:
    print(f"    {child.tag}")

mapsInTheWorld = game.findall("./maps//li")
print(f"There are {len(mapsInTheWorld)} maps in the world.")

# check data type of mapsInTheWorld
# print(type(mapsInTheWorld))

first_child = mapsInTheWorld[0]
print("All tags of the first map: ")
for thing in first_child:
    print(f"    {thing.tag}")

maps/
    li
There are 31412 maps in the world.
All tags of the first map: 
    uniqueID
    generatedId
    generatorDef
    pocketTileInfo
    mapInfo
    layoutStructureSketches
    landingBlockers
    weatherManager
    reservationManager
    enrouteManager
    physicalInteractionReservationManager
    planManager
    designationManager
    pawnDestinationReservationManager
    lordManager
    visitorManager
    gameConditionManager
    fogGrid
    roofGrid
    terrainGrid
    zoneManager
    temperatureCache
    snowGrid
    gasGrid
    pollutionGrid
    waterBodyTracker
    areaManager
    lordsStarter
    attackTargetReservationManager
    deepResourceGrid
    weatherDecider
    damageWatcher
    rememberedCameraPos
    mineStrikeManager
    retainedCaravanData
    storyState
    tempTerrain
    wildPlantSpawner
    temporaryThingDrawer
    flecks
    deferredSpawner
    autoSlaughterManager
    treeDestructionTracker
    storageGroups
    sandGrid
    components
    compressedT

### 9. Convert game ticks to in-game date
- Find the current tick count inside `game/tickManager`.
- RimWorld time: **60,000 ticks = 1 day**, **15 days = 1 quadrum (season)**, **4 quadrums = 1 year**.
- Calculate and print the current **year**, **quadrum (1–4)**, and **day within the quadrum (1–15)**.

In [44]:
tickManager = root.find("game/tickManager")

# check what's in the tickManager
for child in tickManager:
    print(f"{child.tag} -> {child.text}")

ticksGame = root.find("game/tickManager/ticksGame").text

# calculate based on day
TICKS_PER_DAY = 60_000
DAYS_PER_QUADRUM = 15
DAYS_PER_YEAR = DAYS_PER_QUADRUM * 4 # 69 days

ticks = int(ticksGame)
start_year = 5500

year = ticks // (TICKS_PER_DAY * DAYS_PER_YEAR)
ticks %= (TICKS_PER_DAY * DAYS_PER_YEAR)

quadrum = ticks // (TICKS_PER_DAY * DAYS_PER_QUADRUM)
ticks %= (TICKS_PER_DAY * DAYS_PER_QUADRUM)

day = ticks // TICKS_PER_DAY

print(f"Current year: {start_year + year}, quarum: {quadrum}, and day: {day}")

ticksGame -> 28262075
gameStartAbsTick -> 1215000
startingYear -> 5500
Current year: 5507, quarum: 3, and day: 6


### 10. Find all Pawns in the first map
- In RimWorld save files, most elements inside `<things>` have a **`Class` attribute** (e.g. `<li Class="Pawn">`).
- Navigate to the first map's `<things>` tag and find all elements whose `Class` attribute is `"Pawn"`.
- For each pawn found, print the text inside its `<kindDef>` child tag.
- **Hint:** Use `element.get("Class")` to read an XML attribute, or look up XPath attribute syntax for `findall`.

In [ ]:
things = root.find("game/maps/li/things")

# using XPATH
pawns = things.findall(".//thing[@Class='Pawn']")
for pawn in pawns:
    print(pawn.find('kindDef').text)

# using get
for thing in things:
    if thing.get("Class") == "Pawn":
        print(thing.find('kindDef').text)

LabradorRetriever
Boomalope
Boomalope
Colonist
Boomalope
Boomrat
Boomalope
Dromedary
Boomrat
Alpaca
Deer
Colonist
Colonist
Alpaca
Horse
Turkey
WildBoar
WildBoar
Colonist
Rat
Colonist
Colonist
Gazelle
Gazelle
GuineaPig
Turkey
Ibex
Deer
Deer
Husky
Husky
Rhinoceros
Muffalo
Muffalo
Muffalo
Muffalo
Colonist
Colonist
Horse
Rat
Turkey
Hare
Hare
Squirrel
Horse
Horse
Alpaca
Alpaca
Alpaca
Alpaca
GuineaPig
GuineaPig
Horse
Horse
Colonist
Colonist
Colonist
WildBoar
WildBoar
WildBoar
WildBoar
WildBoar
WildBoar
WildBoar
WildBoar
Colonist
Colonist
Horse
Horse
Colonist
Raccoon
Boomrat
Wolf_Timber
Horse
Horse
Colonist
Donkey
Donkey
Colonist
Tribal_Berserker
Tribal_Hunter_Fire
Tribal_Hunter
Tribal_Hunter_Fire
Tribal_Archer
Tribal_Hunter
Tribal_Penitent_Fire
Tribal_Hunter_Fire
Tribal_Hunter
Tribal_Penitent_Fire
Tribal_Warrior_Fire
Tribal_Archer_Fire
Tribal_Archer_Fire
Tribal_HeavyArcher
Tribal_Warrior_Fire
Tribal_Hunter
Tribal_Hunter
Tribal_Hunter_Fire
Tribal_Hunter
Tribal_Hunter_Fire
Tribal_Hunter_Fire
T

### 11. Find all Colonist
- Find all pawns with a text of \<kindDef> is 'Colonist' on the first map's `<things>` and print their last names.

In [49]:
things = root.find("game/maps/li/things")

pawns = things.findall(".//thing[@Class='Pawn'][kindDef='Colonist']")

print("The all last names of our colonists are: ")
for pawn in pawns:
    last_name = pawn.find("name/last").text
    print(last_name)

The all last names of our colonists are: 
크레이그
커즈이스크
크레이그
하이드로넷
하이드로넷
피셔
라우이트
커즈이스크
노턴
하이드로넷
윈'하욘
하이드로넷
호네본
크레이그
플로레스
하이드로넷


### 12. Find all skill levels of Colonists
- Find all skills and levels for each colonist and print them with their first names.

In [60]:
things = root.find("game/maps/li/things")

pawns = things.findall(".//thing[@Class='Pawn'][kindDef='Colonist']")

# check all tags under the pawn
# for pawn in pawns:
#     print(ET.tostring(pawn, encoding='unicode'))
#     break

for pawn in pawns:
    first_name = pawn.find("name/first").text
    print(f"All skill levels of {first_name}: ")
    print(" ")
    skills = pawn.findall("skills/skills/li/def")
    levels = pawn.findall("skills/skills/li/level")
    if len(levels) > 0:
        for skill, level in zip(skills, levels):
            print(f"    {skill.text}, level: {level.text}")
    else:
        print("This colonist is still a baby.")
    print("-" * 30)

All skill levels of 티코: 
 
    Shooting, level: 8
    Melee, level: 1
    Construction, level: 1
    Mining, level: 6
    Cooking, level: 8
    Plants, level: 9
    Animals, level: 1
    Crafting, level: 16
    Artistic, level: 3
    Medicine, level: 4
    Social, level: 7
    Intellectual, level: 15
------------------------------
All skill levels of 나나미: 
 
    Shooting, level: 4
    Melee, level: 8
    Construction, level: 15
    Mining, level: 12
    Cooking, level: 12
    Plants, level: 11
    Animals, level: 2
    Crafting, level: 13
    Artistic, level: 6
    Medicine, level: 2
    Social, level: 2
------------------------------
All skill levels of 앤소니: 
 
    Shooting, level: 11
    Melee, level: 4
    Construction, level: 1
    Mining, level: 1
    Cooking, level: 12
    Plants, level: 3
    Animals, level: 5
    Crafting, level: 1
    Artistic, level: 9
    Medicine, level: 1
------------------------------
All skill levels of 콜린: 
 
    Shooting, level: 5
    Melee, level: 3
 